# Common Track Topo Mode 1 And Mode 2 Demo

This notebook uses the shared `common.topo` namespace to initialize and run the practical Topo Mode 1 and Mode 2 workflows, then inspects the emitted artifacts.

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from common import topo
from topojax.io.imports import load_gmsh_msh

OUTPUT_ROOT = REPO_ROOT / "outputs/common/notebooks/topo_mode12_demo"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT

## 1. Run Topo Mode 1 Through The Common Track

In [ ]:
domain_m1 = topo.initialize_mode1_domain("square", family="tri", nx=8, ny=6, progress=False)
run_m1 = topo.run_mode1_workflow(
    domain_m1,
    output_dir=OUTPUT_ROOT / "mode1",
    prefix="square_tri",
    steps=6,
    step_size=0.02,
    diagnostics_every=3,
    progress=False,
)
metrics_m1 = json.loads(run_m1.artifacts["metrics"].read_text(encoding="utf-8"))
metrics_m1

## 2. Run Topo Mode 2 Through The Common Track

In [ ]:
domain_m2 = topo.initialize_mode2_domain("square", family="tri", nx=8, ny=6, progress=False)
run_m2 = topo.run_mode2_workflow(
    domain_m2,
    output_dir=OUTPUT_ROOT / "mode2",
    prefix="square_tri_restart",
    cycles=1,
    optimization_steps=4,
    optimization_step_size=0.02,
    remesh_max_iters=1,
    progress=False,
)
metrics_m2 = json.loads(run_m2.artifacts["metrics"].read_text(encoding="utf-8"))
phases_m2 = json.loads(run_m2.artifacts["phases"].read_text(encoding="utf-8"))
{"metrics": metrics_m2, "phases": phases_m2}

## 3. Inspect The Exported Mesh Artifacts

In [ ]:
imported_m1 = load_gmsh_msh(run_m1.artifacts["mesh"])
imported_m2 = load_gmsh_msh(run_m2.artifacts["mesh"])
{
    "mode1_primary_kind": imported_m1.primary_element_kind,
    "mode1_nodes": int(imported_m1.points.shape[0]),
    "mode2_primary_kind": imported_m2.primary_element_kind,
    "mode2_nodes": int(imported_m2.points.shape[0]),
    "mode2_artifacts": sorted(run_m2.artifacts),
}